# MP05 — Gaussian Blur Simulator

Expanded engineering lesson; learner mastery unassessed.

## Intuition, definitions and worked mathematics
A blur replaces each sample by a weighted neighborhood. H and W count rows and columns; C counts bands. Our image is H×W×C in floating numerical units. For integer offsets i,j in [−r,r], the Gaussian weight is exp(−(i²+j²)/(2 sigma²))/Z. Sigma is a positive width in pixels, r is an integer radius, and Z is the sum of all retained weights. The support has width 2r+1. Normalization makes the weights sum to one, so a constant image stays constant under replicated or reflected boundaries.

The implementation computes correlation: y[h,w,c]=sum(k[i,j]x[h+i,w+j,c]). Convolution reverses the kernel offsets. For a symmetric Gaussian they agree; they need not agree for an asymmetric edge kernel. Edge replication copies the nearest pixel; reflection mirrors nearby values; constant padding inserts zeros. These assumptions affect edge pixels and must be recorded.

For sigma=1 and radius=1, the unnormalized 1D weights are approximately [0.60653,1,0.60653], normalized to [0.27407,0.45186,0.27407]. Their outer product gives the 3×3 kernel, with center about 0.20418. A centered unit impulse produces the kernel response. Greater sigma spreads influence only if the support is large enough; truncating to radius one cannot represent a very broad Gaussian.

In [ ]:
from pathlib import Path
import sys,json
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(root/'src'))
from mp05_tools import gaussian,filter_hwc,degrade_hwc,spectrum
k=gaussian(1.,1)
print(k,k.sum())
assert np.isclose(k[1,1],.2041799556)
impulse=np.zeros((9,9,1)); impulse[4,4]=1
y=filter_hwc(impulse,k)
np.testing.assert_allclose(y[3:6,3:6,0],k)
constant=np.ones((9,9,1))
print('edge/zero-padding corners:',filter_hwc(constant,k)[0,0,0],filter_hwc(constant,k,'constant')[0,0,0])

## Real-data experiment and interpretation
The real experiment sweeps (sigma,radius)=(0.5,2),(1,3),(2,6), plots kernels and impulse/step-edge responses, and measures how much a fixed training crop changes. The corner of a constant image is preserved by replication but darkens with zero padding. This is a boundary artifact, not evidence of a dark object. Blur MSE against the original quantifies a change, not whether blur is scientifically appropriate.

Run `python scripts/degradation_experiments.py` explicitly from the repository root. The notebook below reads recorded measurements; it does not download data or execute a hidden research run. The source is a checksummed 64×64 NAIP training crop with fixed stored-value/10000 scaling, not verified reflectance.

In [ ]:
p=root/'reports/mp05-mp08-experiments.json'
if p.exists():
    record=json.loads(p.read_text())
    print(record['mp05'])
else: print('Real experiment has not run in this checkout.')

In [ ]:
from IPython.display import Image,display
p=root/'reports'/'mp05-kernels.png'
if p.exists(): display(Image(filename=str(p)))

## Practice — attempt before reading solutions
1. Why does a normalized blur preserve a constant image?
2. What happens when radius=0?
3. Why might sigma=5 with radius=1 be misleading?

## Separate solution guide
1. The weights sum to one, so weighted copies of the same constant sum to that constant, provided padding preserves it.
2. There is only one weight, equal to one, so the image is unchanged.
3. Most Gaussian mass is omitted; renormalization creates a short nearly flat kernel, not a broad sensor point-spread function.

These are generated examples, not evidence that a learner solved the exercises.

In [ ]:
np.testing.assert_array_equal(filter_hwc(np.arange(18).reshape(2,3,3),gaussian(1,0)),np.arange(18).reshape(2,3,3))

## Reuse, execution and limitations
Use `src/rssr/degradation_lab.py` and `python -m pytest tests/test_degradation_lab.py`. Follow [local and Colab setup](../../docs/setup.md); run from notebooks/ or the repository root. The shared NumPy code preserves HWC channels and uses explicit boundaries. No physical sensor calibration or geographic generalization is established.

References: the authoritative stage brief, [dataset card](../../docs/dataset-cards.md), [protocol](../../docs/protocol.md), and [NumPy FFT documentation](https://numpy.org/doc/stable/reference/routines.fft.html). See [execution evidence](../../reports/notebook-execution.json) and [experiment record](../../reports/mp05-mp08-experiments.json).